# Notebook 2: TerraClimate Feature Extraction

## Purpose
Extracts 14 monthly climate variables (temperature, precipitation, soil moisture, evapotranspiration, etc.) for all 9,319 water quality samples from the TerraClimate dataset via Microsoft Planetary Computer.

## Input
- `unique_samples_for_extraction.csv` (9,319 unique location-date combinations)

## Process
- Spatial matching: k-d tree finds nearest 4km TerraClimate grid cell for each sample location
- Temporal matching: Sample date matched to corresponding month (e.g., March 15, 2014 → March 2014 grid)

## Output
- `terraclimate_features.csv` (9,319 rows × 17 columns: Lat, Lon, Date, 14 climate variables)

## Cell 1: Setup & Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import xarray as xr
from scipy.spatial import cKDTree
import pystac_client
import planetary_computer as pc
from tqdm import tqdm

# Dask client for parallel processing
from dask.distributed import Client

print("Libraries loaded successfully!")

Libraries loaded successfully!


## Cell 2: Load Unique Samples for Extraction

In [3]:
# Load the unique sample list created in Notebook 1
df_samples = pd.read_csv('unique_samples_for_extraction.csv')

print("="*80)
print("UNIQUE SAMPLES LOADED")
print("="*80)
print(f"\nTotal samples to extract: {len(df_samples):,}")
print(f"Columns: {list(df_samples.columns)}")

# Parse dates
df_samples['Sample Date'] = pd.to_datetime(df_samples['Sample Date'])

print(f"\nDate range:")
print(f"  Earliest: {df_samples['Sample Date'].min()}")
print(f"  Latest: {df_samples['Sample Date'].max()}")

# Count unique locations
unique_locs = df_samples[['Latitude', 'Longitude']].drop_duplicates()
print(f"\nUnique locations: {len(unique_locs):,}")

print(f"\nFirst few rows:")
display(df_samples.head())

UNIQUE SAMPLES LOADED

Total samples to extract: 9,319
Columns: ['Latitude', 'Longitude', 'Sample Date']

Date range:
  Earliest: 2011-01-02 00:00:00
  Latest: 2015-12-31 00:00:00

Unique locations: 162

First few rows:


,Latitude,Longitude,Sample Date
0,-34.405833,19.600556,2011-07-06
1,-34.405833,19.600556,2011-08-02
2,-34.405833,19.600556,2011-08-30
3,-34.405833,19.600556,2011-09-27
4,-34.405833,19.600556,2011-10-27


## Cell 3: Define TerraClimate Variables

In [5]:
# TerraClimate variables to extract
tc_variables = [
    'pet',   # Potential evapotranspiration (mm)
    'aet',   # Actual evapotranspiration (mm)
    'ppt',   # Precipitation (mm)
    'tmax',  # Maximum temperature (°C)
    'tmin',  # Minimum temperature (°C)
    'q',     # Runoff (mm)
    'soil',  # Soil moisture (mm)
    'swe',   # Snow water equivalent (mm)
    'def',   # Climate water deficit (mm)
    'pdsi',  # Palmer Drought Severity Index
    'vap',   # Vapor pressure (kPa)
    'vpd',   # Vapor pressure deficit (kPa)
    'ws',    # Wind speed (m/s)
    'srad'   # Solar radiation (W/m²)
]

print("="*80)
print("TERRACLIMATE VARIABLES TO EXTRACT")
print("="*80)
print(f"\nTotal variables: {len(tc_variables)}")
for i, var in enumerate(tc_variables, 1):
    print(f"  {i:2d}. {var}")

TERRACLIMATE VARIABLES TO EXTRACT

Total variables: 14
   1. pet
   2. aet
   3. ppt
   4. tmax
   5. tmin
   6. q
   7. soil
   8. swe
   9. def
  10. pdsi
  11. vap
  12. vpd
  13. ws
  14. srad


## Cell 4: Define Optimized Extraction Functions

In [27]:
def extract_terraclimate_optimized(ds, variables, sample_locations):
    """
    Extract TerraClimate data by loading each month's full SA grid into memory.
    Includes token refresh to handle long extractions.
    """
    results = []
    unique_locs = sample_locations[['Latitude', 'Longitude']].drop_duplicates().reset_index(drop=True)
    
    print(f"Extracting {len(variables)} variables at {len(unique_locs)} unique locations for {len(ds.time)} months...")
    print(f"Loading full SA grid per month (~50MB/month)...")
    print(f"Expected time: ~1.5-2 hours (with token refresh every 10 months)\n")
    
    for time_idx in tqdm(range(len(ds.time)), desc="Processing months"):
        
        # REFRESH TOKEN EVERY 10 MONTHS (~15 minutes)
        if time_idx > 0 and time_idx % 10 == 0:
            print(f"\nRefreshing token at month {time_idx}...")
            catalog = pystac_client.Client.open(
                "https://planetarycomputer.microsoft.com/api/stac/v1",
                modifier=pc.sign_inplace)
            collection = catalog.get_collection("terraclimate")
            asset = collection.assets["zarr-abfs"]
            ds_refreshed = xr.open_dataset(asset.href, **asset.extra_fields["xarray:open_kwargs"])
            ds_refreshed = ds_refreshed.drop('crs', dim=None, errors='ignore')
            ds_refreshed = ds_refreshed.sel(time=slice("2011-01-01", "2015-12-31"))
            mask_lon = (ds_refreshed.lon >= 14.97) & (ds_refreshed.lon <= 32.79)
            mask_lat = (ds_refreshed.lat >= -35.18) & (ds_refreshed.lat <= -21.72)
            ds = ds_refreshed.where(mask_lon & mask_lat, drop=True)
            print("Token refreshed!\n")
        
        # Load ENTIRE SA grid for this month into memory
        time_slice = ds[variables].isel(time=time_idx).load()
        current_date = pd.Timestamp(time_slice.time.values)
        
        # Extract from memory (fast)
        for idx, loc in unique_locs.iterrows():
            lat, lon = loc['Latitude'], loc['Longitude']
            point = time_slice.sel(lat=lat, lon=lon, method='nearest')
            
            record = {
                'Latitude': float(point.lat.values),
                'Longitude': float(point.lon.values),
                'Sample Date': current_date
            }
            
            for var in variables:
                record[var] = float(point[var].values)
            
            results.append(record)
    
    df_result = pd.DataFrame(results)
    print(f"\n✓ Extracted {len(df_result):,} records")
    return df_result

def assign_climate_by_month(sample_df, climate_df, variables):
    """
    Assign climate values by matching location and month/year.
    Optimized for monthly TerraClimate data.
    
    Parameters:
    -----------
    sample_df : pd.DataFrame
        Sample data with 'Latitude', 'Longitude', 'Sample Date'
    climate_df : pd.DataFrame
        Climate data with same columns plus variable columns
    variables : list
        List of climate variable names to assign
    
    Returns:
    --------
    pd.DataFrame with climate values assigned to samples
    """
    sample_df = sample_df.copy().reset_index(drop=True)
    climate_df = climate_df.copy()
    
    # Parse dates
    sample_df['Sample Date'] = pd.to_datetime(sample_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], errors='coerce')
    
    # Add month/year columns for faster matching
    sample_df['year_month'] = sample_df['Sample Date'].dt.to_period('M')
    climate_df['year_month'] = climate_df['Sample Date'].dt.to_period('M')
    
    # Build spatial index using KD-tree
    print("Building spatial index...")
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)
    tree = cKDTree(climate_coords)
    
    results = []
    
    print("Matching samples to climate data...")
    for i in tqdm(range(len(sample_df)), desc="Assigning values"):
        sample_row = sample_df.iloc[i]
        sample_coords = np.radians([[sample_row['Latitude'], sample_row['Longitude']]])
        sample_year_month = sample_row['year_month']
        
        # Find nearest climate location (spatial matching)
        dist, idx = tree.query(sample_coords, k=1)
        nearest_climate = climate_df.iloc[idx[0]]
        
        # Filter to same location AND same month/year (temporal matching)
        matches = climate_df[
            (climate_df['Latitude'] == nearest_climate['Latitude']) &
            (climate_df['Longitude'] == nearest_climate['Longitude']) &
            (climate_df['year_month'] == sample_year_month)
        ]
        
        if not matches.empty:
            record = {
                'Latitude': sample_row['Latitude'],
                'Longitude': sample_row['Longitude'],
                'Sample Date': sample_row['Sample Date']
            }
            for var in variables:
                record[var] = matches.iloc[0][var]
            results.append(record)
        else:
            # No match found - append NaNs
            record = {
                'Latitude': sample_row['Latitude'],
                'Longitude': sample_row['Longitude'],
                'Sample Date': sample_row['Sample Date']
            }
            for var in variables:
                record[var] = np.nan
            results.append(record)
    
    result_df = pd.DataFrame(results)
    
    # Drop temporary year_month column from sample_df
    sample_df.drop('year_month', axis=1, inplace=True)
    
    print(f"\n✓ Matched {len(result_df):,} samples")
    print(f"\nMissing values per variable:")
    print(result_df[variables].isna().sum())
    
    return result_df

print("Functions defined successfully!")

Functions defined successfully!


## Cell 5: Initialize Planetary Computer & Load TerraClimate Dataset

In [29]:
print("Loading terraclimate dataset")

# Initialize Dask client
client = Client(memory_limit='4GB', processes=False)
print(f"Dask client initialized: {client}")

# Connect to Planetary Computer STAC catalog
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace)

# Get TerraClimate collection
collection = catalog.get_collection("terraclimate")
asset = collection.assets["zarr-abfs"]

# Load dataset
print("\nLoading TerraClimate dataset...")
ds = xr.open_dataset(asset.href, **asset.extra_fields["xarray:open_kwargs"])
ds = ds.drop('crs', dim=None, errors='ignore')

# Select time range (2011-2015)
ds = ds.sel(time=slice("2011-01-01", "2015-12-31"))

# Trim to South Africa bounding box
print("Trimming to South Africa bounding box...")
mask_lon = (ds.lon >= 14.97) & (ds.lon <= 32.79)
mask_lat = (ds.lat >= -35.18) & (ds.lat <= -21.72)
ds = ds.where(mask_lon & mask_lat, drop=True)

print("\nDataset loaded and trimmed!")
print(f"Time range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Timesteps: {len(ds.time)}")
print(f"Grid size: {len(ds.lat)} lat × {len(ds.lon)} lon")
print(f"Variables: {tc_variables}")
print(f"\nDataset info:")
print(ds)

Loading terraclimate dataset
Dask client initialized: <Client: 'inproc://149.125.193.81/26792/36' processes=1 threads=8, memory=3.73 GiB>

Loading TerraClimate dataset...
Trimming to South Africa bounding box...

Dataset loaded and trimmed!
Time range: 2011-01-01T00:00:00.000000000 to 2015-12-01T00:00:00.000000000
Timesteps: 60
Grid size: 323 lat × 428 lon
Variables: ['pet', 'aet', 'ppt', 'tmax', 'tmin', 'q', 'soil', 'swe', 'def', 'pdsi', 'vap', 'vpd', 'ws', 'srad']

Dataset info:
<xarray.Dataset> Size: 564MB
Dimensions:  (time: 60, lat: 323, lon: 428)
Coordinates:
  * time     (time) datetime64[ns] 480B 2011-01-01 2011-02-01 ... 2015-12-01
  * lat      (lat) float64 3kB -21.73 -21.77 -21.81 ... -35.06 -35.1 -35.15
  * lon      (lon) float64 3kB 14.98 15.02 15.06 15.1 ... 32.69 32.73 32.77
Data variables: (12/14)
    aet      (time, lat, lon) float32 33MB dask.array<chunksize=(12, 323, 428), meta=np.ndarray>
    def      (time, lat, lon) float32 33MB dask.array<chunksize=(12, 323, 428)

## Cell 6: Extract TerraClimate Data at Sample Locations

In [31]:
print("="*80)
print("EXTRACTING TERRACLIMATE DATA")
print("="*80)
print(f"\nExtracting {len(tc_variables)} variables for {len(df_samples):,} samples")
print(f"Unique locations: {df_samples[['Latitude', 'Longitude']].drop_duplicates().shape[0]:,}")
print(f"Time period: 2011-2015 ({len(ds.time)} months)")
print("\nThis will extract data ONLY at sample locations (not full grid) - much faster!\n")

# Extract climate data at sample locations only
tc_all_filtered = extract_terraclimate_optimized(ds, tc_variables, df_samples)

print(f"\nExtracted climate data shape: {tc_all_filtered.shape}")
print(f"Date range: {tc_all_filtered['Sample Date'].min()} to {tc_all_filtered['Sample Date'].max()}")
print(f"\nFirst few rows:")
display(tc_all_filtered.head())

EXTRACTING TERRACLIMATE DATA

Extracting 14 variables for 9,319 samples
Unique locations: 162
Time period: 2011-2015 (60 months)

This will extract data ONLY at sample locations (not full grid) - much faster!

Extracting 14 variables at 162 unique locations for 60 months...
Loading full SA grid per month (~50MB/month)...
Expected time: ~1.5-2 hours (with token refresh every 10 months)



Processing months:  17%|███▌                 | 10/60 [20:30<1:36:30, 115.82s/it]


Refreshing token at month 10...
Token refreshed!



Processing months:  33%|███████▎              | 20/60 [36:05<1:03:24, 95.11s/it]


Refreshing token at month 20...
Token refreshed!



Processing months:  42%|██████████              | 25/60 [44:17<57:29, 98.56s/it]2026-04-30 17:53:29,848 - distributed.worker - ERROR - Compute Failed
Key:       ('open_dataset-vpd-getitem-0a7c58fa9082e0ff2e33d59b9e7b40af', 2, 2, 4)
State:     executing
Task:  <Task ('open_dataset-vpd-getitem-0a7c58fa9082e0ff2e33d59b9e7b40af', 2, 2, 4) _execute_subgraph(...)>
Exception: "ClientAuthenticationError('Server failed to authenticate the request. Make sure the value of Authorization header is formed correctly including the signature.\\nRequestId:f4d48fa3-401e-006b-20eb-d88da6000000\\nTime:2026-04-30T21:53:29.4831098Z\\nErrorCode:AuthenticationFailed\\nauthenticationerrordetail:Signature not valid in the specified time frame: Start [Wed, 29 Apr 2026 21:08:06 GMT] - Expiry [Thu, 30 Apr 2026 21:53:06 GMT] - Current [Thu, 30 Apr 2026 21:53:29 GMT]')"
Traceback: '  File "/opt/anaconda3/lib/python3.12/site-packages/dask/array/core.py", line 140, in getter\n    c = np.asarray(c)\n        ^^^^^^^^^^^^

ClientAuthenticationError: Server failed to authenticate the request. Make sure the value of Authorization header is formed correctly including the signature.
RequestId:f4d48fa3-401e-006b-20eb-d88da6000000
Time:2026-04-30T21:53:29.4831098Z
ErrorCode:AuthenticationFailed
authenticationerrordetail:Signature not valid in the specified time frame: Start [Wed, 29 Apr 2026 21:08:06 GMT] - Expiry [Thu, 30 Apr 2026 21:53:06 GMT] - Current [Thu, 30 Apr 2026 21:53:29 GMT]
Content: <?xml version="1.0" encoding="utf-8"?><Error><Code>AuthenticationFailed</Code><Message>Server failed to authenticate the request. Make sure the value of Authorization header is formed correctly including the signature.
RequestId:f4d48fa3-401e-006b-20eb-d88da6000000
Time:2026-04-30T21:53:29.4831098Z</Message><AuthenticationErrorDetail>Signature not valid in the specified time frame: Start [Wed, 29 Apr 2026 21:08:06 GMT] - Expiry [Thu, 30 Apr 2026 21:53:06 GMT] - Current [Thu, 30 Apr 2026 21:53:29 GMT]</AuthenticationErr

## Cell 7: Match Climate Data to Sample Dates

In [ ]:
print("="*80)
print("MATCHING CLIMATE DATA TO SAMPLE DATES")
print("="*80)
print("\nMatching by nearest location AND month/year...\n")

# Match samples to climate data by location and month/year
terraclimate_df = assign_climate_by_month(df_samples, tc_all_filtered, tc_variables)

# Keep only relevant columns
output_cols = ['Latitude', 'Longitude', 'Sample Date'] + tc_variables
terraclimate_df = terraclimate_df[output_cols]

print(f"\nMatching complete!")
print(f"Final shape: {terraclimate_df.shape}")
print(f"Expected: ({len(df_samples)}, {len(output_cols)})")

print(f"\nFirst 10 rows:")
display(terraclimate_df.head(10))

## Cell 8: Data Quality Checks

In [ ]:
# Quality checks
print("DATA QUALITY CHECKS")
print(f"Row count: {len(terraclimate_df)} / {len(df_samples)} expected")
print(f"Duplicates: {terraclimate_df.duplicated(subset=['Latitude', 'Longitude', 'Sample Date']).sum()}")
print(f"\nMissing values:\n{terraclimate_df[tc_variables].isna().sum()}")
print(f"\nValue ranges:\n{terraclimate_df[tc_variables].describe()}")

## Cell 9: Save Output

In [ ]:
import base64
from IPython.display import HTML, display

# Save to CSV
terraclimate_df.to_csv('terraclimate_features.csv', index=False)

print(f"Saved: terraclimate_features.csv ({terraclimate_df.shape})")
print(f"Missing values: {terraclimate_df[tc_variables].isna().sum().sum()}")

# Download link
csv = terraclimate_df.to_csv(index=False)
b64 = base64.b64encode(csv.encode()).decode()
display(HTML(f'<a href="data:file/csv;base64,{b64}" '
            f'download="terraclimate_features.csv">Download terraclimate_features.csv</a>'))